# S4 · AndinaLog 03B · Notebook 2 · Tratamiento de Flota

Este notebook lee el diagnóstico v2 de `andinalog_flota.csv` y el catálogo de reglas de Flota. Conserva las 32 filas Bronze en el archivo tratado y genera un Silver solo con registros utilizables. No modifica el Bronze ni aplica una regla que esté `PENDIENTE`.

El catálogo inicial mantiene todas las reglas pendientes. Cada aprobación requiere un acuerdo documentado en `evidencia_acuerdo` antes de ejecutar el tratamiento.


## 1 · Rutas y entradas

En local, ejecuta dentro de `practicasNotebookColab`. En Colab, ajusta `RUTA_PROYECTO_DRIVE` a la carpeta con `datasets/` y `proyecto-integrador/`. Coloca `catalogo_reglas_tratamiento_flota.csv` en `proyecto-integrador/andinalog_flota/notebook2/`.


In [2]:
from pathlib import Path
import hashlib
import os
import tempfile
import pandas as pd

ENTORNO = "auto"  # auto, local o drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
VERSION_DIAGNOSTICO_REQUERIDA = "GIAD-M3-S4-FLOTA-diagnostico-v2"
VERSION_TRATAMIENTO = "GIAD-M3-S4-FLOTA-tratamiento-v1"
COLUMNAS_BRONZE = ["camion_id", "centro_distribucion_base", "capacidad_kg", "tipo_camion"]

def raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if ((carpeta / "datasets" / "AndinaLog_03B_Bronce").is_dir()
                and (carpeta / "proyecto-integrador").is_dir()):
            return carpeta
    raise FileNotFoundError("Ejecuta dentro de practicasNotebookColab")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "auto":
        entorno = "drive" if "google.colab" in __import__("sys").modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = raiz_local()
    else:
        raise ValueError("ENTORNO debe ser auto, local o drive")
    caso = raiz / "proyecto-integrador" / "andinalog_flota"
    return {
        "bronze": raiz / "datasets" / "AndinaLog_03B_Bronce" / "andinalog_flota.csv",
        "principal": caso / "notebook1" / "salidas" / "andinalog_flota_diagnosticado.csv",
        "problemas": caso / "notebook1" / "salidas" / "andinalog_flota_problemas.csv",
        "reporte1": caso / "notebook1" / "salidas" / "andinalog_flota_reporte_calidad.csv",
        "catalogo": caso / "notebook2" / "catalogo_reglas_tratamiento_flota.csv",
        "salidas": caso / "notebook2" / "salidas",
    }

rutas = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
for nombre in ["bronze", "principal", "problemas", "reporte1", "catalogo"]:
    if not rutas[nombre].is_file():
        raise FileNotFoundError(f"Falta {nombre}: {rutas[nombre]}")
print("Bronze:", rutas["bronze"])
print("Salidas:", rutas["salidas"])


Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_flota.csv
Salidas: c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_flota\notebook2\salidas


## 2 · Lectura y validación

Se exige que el reporte del notebook 1 corresponda al Bronze actual por SHA-256 y versión. Si falla, vuelve a ejecutar el diagnóstico.


In [3]:
def leer_csv(ruta):
    return pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)

bronze = leer_csv(rutas["bronze"])
principal = leer_csv(rutas["principal"])
problemas = leer_csv(rutas["problemas"])
reporte1 = leer_csv(rutas["reporte1"])
catalogo = leer_csv(rutas["catalogo"])
if list(bronze.columns) != COLUMNAS_BRONZE:
    raise ValueError("Esquema Bronze inesperado")
if not {"fila_bronze", *COLUMNAS_BRONZE, "en_cuarentena"}.issubset(principal.columns):
    raise ValueError("Diagnosticado incompleto")
if not {"fila_bronze", "columna_afectada", "codigo_error", "valor_original", "version_diagnostico"}.issubset(problemas.columns):
    raise ValueError("Detalle de problemas incompleto")
if not {"regla_id", "estado", "evidencia_acuerdo"}.issubset(catalogo.columns):
    raise ValueError("Catálogo incompleto")
if principal["fila_bronze"].duplicated().any() or catalogo["regla_id"].duplicated().any():
    raise ValueError("Identificadores internos duplicados")
if not catalogo["estado"].isin(["APROBADA", "PENDIENTE"]).all():
    raise ValueError("Estado de regla desconocido")
if not problemas["fila_bronze"].isin(principal["fila_bronze"]).all():
    raise ValueError("Problemas sin fila diagnosticada")
resumen = reporte1.set_index("metrica")["valor"]
HASH_BRONZE = hashlib.sha256(rutas["bronze"].read_bytes()).hexdigest()
if resumen.loc["sha256_bronze"] != HASH_BRONZE:
    raise ValueError("Bronze distinto del diagnóstico; ejecuta notebook 1")
if resumen.loc["version_diagnostico"] != VERSION_DIAGNOSTICO_REQUERIDA:
    raise ValueError("Se requiere diagnóstico v2 de Flota")
if len(bronze) != len(principal) or len(bronze) != int(resumen.loc["filas_bronze"]):
    raise ValueError("Conteo de filas incompatible")
if len(problemas) != int(resumen.loc["problemas_detectados"]):
    raise ValueError("Conteo de problemas incompatible")
pd.testing.assert_frame_equal(principal[COLUMNAS_BRONZE].reset_index(drop=True), bronze.reset_index(drop=True))
if set(problemas["fila_bronze"]) != set(principal.loc[principal["en_cuarentena"].str.lower().eq("true"), "fila_bronze"]):
    raise ValueError("Cuarentena y problemas no coinciden")
print(f"Entradas válidas: {len(bronze)} filas; {len(problemas)} problemas; SHA-256 {HASH_BRONZE}")
display(catalogo)


Entradas válidas: 32 filas; 4 problemas; SHA-256 5da0620ea1883c3bad57ba31f45c24b454fe3dadede3cf2370eff459714fbf14


,regla_id,columna_afectada,codigo_error,tratamiento_propuesto,estado,evidencia_acuerdo,validacion_requerida
0,ID_NORMALIZAR,camion_id,FORMATO_INVALIDO,Quitar espacios exteriores y pasar a mayúscula...,PENDIENTE,,Aprobar equivalencia; revisar colisiones despu...
1,DUPLICADO_IDENTICO,camion_id,DUPLICADO_IDENTICO,Conservar la primera fila y excluir copias con...,PENDIENTE,,Aprobar criterio de canonicidad y comprobar ig...
2,COLISION_ID_NORMALIZADO,camion_id,COLISION_ID_NORMALIZADO,Excluir copia con ID equivalente solo si los o...,PENDIENTE,,Requiere ID_NORMALIZAR aprobada y comparación ...
3,ID_EN_CONFLICTO,camion_id,ID_EN_CONFLICTO,Mantener todas las filas en cuarentena hasta v...,PENDIENTE,,Determinar registro autoritativo; no elegir au...
4,CENTRO_FALTANTE,centro_distribucion_base,FALTANTE,Mantener en cuarentena hasta recuperar el centro,PENDIENTE,,Centro confirmado por fuente autorizada
5,CAPACIDAD_FALTANTE,capacidad_kg,FALTANTE,Mantener en cuarentena sin imputar capacidad,PENDIENTE,,Capacidad confirmada por ficha de flota
6,CAPACIDAD_NO_NUMERICA,capacidad_kg,NO_NUMERICA,Mantener en cuarentena hasta recuperar valor n...,PENDIENTE,,Capacidad confirmada por ficha de flota
7,CAPACIDAD_NO_POSITIVA,capacidad_kg,NO_POSITIVA,Mantener en cuarentena hasta verificar capacid...,PENDIENTE,,Capacidad confirmada por ficha de flota
8,TIPO_FALTANTE,tipo_camion,FALTANTE,Mantener en cuarentena hasta recuperar tipo,PENDIENTE,,Tipo confirmado por fuente autorizada
9,TIPO_NO_RECONOCIDO,tipo_camion,TIPO_NO_RECONOCIDO,Mantener en cuarentena hasta validar categoría,PENDIENTE,,Categoría confirmada por fuente autorizada


## 3 · Reglas y decisiones de duplicados

Los identificadores se normalizan únicamente con aprobación. Las copias idénticas se pueden excluir según su propia regla. Si dos registros comparten ID pero discrepan en otros campos, permanecen pendientes. Las colisiones que aparecerían al normalizar requieren aprobación adicional y comparación de los otros tres campos.


In [4]:
def aprobada(regla_id):
    fila = catalogo.loc[catalogo["regla_id"].eq(regla_id)]
    if len(fila) != 1:
        raise ValueError(f"Falta regla única: {regla_id}")
    ok = fila.iloc[0]["estado"] == "APROBADA"
    if ok and not str(fila.iloc[0]["evidencia_acuerdo"]).strip():
        raise ValueError(f"Regla {regla_id} aprobada sin evidencia")
    return ok

AUTORIZADAS = {rid: aprobada(rid) for rid in ["ID_NORMALIZAR", "DUPLICADO_IDENTICO", "COLISION_ID_NORMALIZADO"]}
df_trabajo = principal.copy(deep=True)
id_original = df_trabajo["camion_id"]
id_candidato = id_original.str.strip().str.upper()
df_trabajo["camion_id_preparado"] = id_original.copy()
if AUTORIZADAS["ID_NORMALIZAR"]:
    valido = id_candidato.str.fullmatch(r"CAM-\d{2}").fillna(False)
    cambiar = valido & id_original.ne(id_candidato)
    df_trabajo.loc[cambiar, "camion_id_preparado"] = id_candidato[cambiar]

def decidir_duplicados(df):
    copia = df.copy()
    copia["_id_candidato"] = copia["camion_id"].str.strip().str.upper()
    decisiones = []
    for clave, grupo in copia.groupby("_id_candidato", sort=False):
        if len(grupo) < 2:
            continue
        grupo = grupo.sort_values("fila_bronze", key=lambda s: s.astype(int))
        exacto = grupo[COLUMNAS_BRONZE].nunique(dropna=False).eq(1).all()
        otros_iguales = grupo[[c for c in COLUMNAS_BRONZE if c != "camion_id"]].nunique(dropna=False).eq(1).all()
        if exacto:
            tipo, autorizado = "IDENTICO", AUTORIZADAS["DUPLICADO_IDENTICO"]
        elif otros_iguales:
            tipo = "ID_EQUIVALENTE"
            autorizado = AUTORIZADAS["ID_NORMALIZAR"] and AUTORIZADAS["COLISION_ID_NORMALIZADO"]
        else:
            tipo, autorizado = "CONFLICTO", False
        canonica = grupo.iloc[0]["fila_bronze"] if autorizado else ""
        for _, fila in grupo.iterrows():
            decision = ("CANONICA" if fila["fila_bronze"] == canonica else "COPIA_EXCLUIDA") if autorizado else "PENDIENTE"
            decisiones.append({"fila_bronze": fila["fila_bronze"], "camion_id_normalizado": clave,
                               "tipo_duplicado": tipo, "decision_duplicado": decision,
                               "fila_canonica": canonica,
                               "justificacion": "Campos coincidentes y regla aprobada" if autorizado else "Sin regla aprobada o datos en conflicto"})
    return pd.DataFrame(decisiones, columns=["fila_bronze", "camion_id_normalizado", "tipo_duplicado",
                                             "decision_duplicado", "fila_canonica", "justificacion"])

decisiones = decidir_duplicados(df_trabajo)
display(decisiones)


,fila_bronze,camion_id_normalizado,tipo_duplicado,decision_duplicado,fila_canonica,justificacion
0,1,CAM-01,IDENTICO,PENDIENTE,,Sin regla aprobada o datos en conflicto
1,31,CAM-01,IDENTICO,PENDIENTE,,Sin regla aprobada o datos en conflicto
2,27,CAM-27,IDENTICO,PENDIENTE,,Sin regla aprobada o datos en conflicto
3,32,CAM-27,IDENTICO,PENDIENTE,,Sin regla aprobada o datos en conflicto


## 4 · Acciones, cuarentena final y Silver

Solo una acción resuelta deja de ser motivo de cuarentena. Las copias excluidas se conservan en el archivo completo para auditoría y quedan fuera del Silver.


In [5]:
acciones = problemas.copy(deep=True)
acciones["estado_tratamiento"] = "PENDIENTE"
acciones["tratamiento_aplicado"] = "NINGUNO"
acciones["detalle_resultado"] = "Sin regla aprobada o evidencia suficiente"
por_fila = df_trabajo.set_index("fila_bronze")
if AUTORIZADAS["ID_NORMALIZAR"]:
    preparado = acciones["fila_bronze"].map(por_fila["camion_id_preparado"])
    original = acciones["fila_bronze"].map(por_fila["camion_id"])
    normalizado = (acciones["codigo_error"].eq("FORMATO_INVALIDO")
                   & preparado.str.fullmatch(r"CAM-\d{2}").fillna(False)
                   & preparado.ne(original))
    acciones.loc[normalizado, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
        "RESUELTO", "ID_NORMALIZADO", "ID normalizado según regla aprobada"]
por_decision = decisiones.set_index("fila_bronze")["decision_duplicado"]
decision = acciones["fila_bronze"].map(por_decision)
dup = acciones["codigo_error"].isin(["DUPLICADO_IDENTICO", "COLISION_ID_NORMALIZADO", "ID_EN_CONFLICTO"])
canonica = dup & decision.eq("CANONICA")
copia = dup & decision.eq("COPIA_EXCLUIDA")
acciones.loc[canonica, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
    "RESUELTO", "SELECCION_CANONICA", "Fila canónica según regla aprobada"]
acciones.loc[copia, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = [
    "EXCLUIDO_COMO_COPIA", "COPIA_EXCLUIDA", "Copia conservada para auditoría"]
marcadas = set(acciones.loc[copia, "fila_bronze"])
extra = decisiones.loc[decisiones["decision_duplicado"].eq("COPIA_EXCLUIDA")
                      & ~decisiones["fila_bronze"].isin(marcadas)]
if len(extra):
    adicionales = pd.DataFrame({"fila_bronze": extra["fila_bronze"],
        "columna_afectada": "camion_id", "codigo_error": "COPIA_EXCLUIDA",
        "valor_original": extra["camion_id_normalizado"],
        "version_diagnostico": VERSION_DIAGNOSTICO_REQUERIDA,
        "estado_tratamiento": "EXCLUIDO_COMO_COPIA", "tratamiento_aplicado": "COPIA_EXCLUIDA",
        "detalle_resultado": "Copia conservada para auditoría"})
    acciones = pd.concat([acciones, adicionales], ignore_index=True)
acciones = acciones.sort_values(["fila_bronze", "codigo_error"],
                                key=lambda s: s.astype(int) if s.name == "fila_bronze" else s,
                                kind="stable").reset_index(drop=True)

df_final = df_trabajo.copy(deep=True)
for campo in ["tipo_duplicado", "decision_duplicado", "fila_canonica", "justificacion"]:
    df_final[campo] = df_final["fila_bronze"].map(decisiones.set_index("fila_bronze")[campo]).fillna("")
df_final["en_cuarentena_inicial"] = df_final["en_cuarentena"].str.lower().eq("true")
pendientes = acciones.loc[acciones["estado_tratamiento"].ne("RESUELTO")].copy()
pendientes["motivo"] = pendientes["columna_afectada"] + ":" + pendientes["codigo_error"]
motivos = pendientes.groupby("fila_bronze")["motivo"].agg(lambda x: "|".join(dict.fromkeys(x)))
df_final["motivos_finales"] = df_final["fila_bronze"].map(motivos).fillna("")
df_final["en_cuarentena_final"] = df_final["motivos_finales"].ne("")
df_final["version_tratamiento"] = VERSION_TRATAMIENTO
cuarentena_final = df_final.loc[df_final["en_cuarentena_final"]].copy()
silver = df_final.loc[~df_final["en_cuarentena_final"],
                      ["camion_id_preparado", "centro_distribucion_base", "capacidad_kg", "tipo_camion"]].copy()
silver.columns = COLUMNAS_BRONZE
silver["capacidad_kg"] = pd.to_numeric(silver["capacidad_kg"].str.strip(), errors="coerce")

pd.testing.assert_frame_equal(df_final[COLUMNAS_BRONZE].reset_index(drop=True), bronze.reset_index(drop=True))
assert len(df_final) == len(bronze)
assert len(silver) + len(cuarentena_final) == len(bronze)
assert not silver["camion_id"].duplicated().any()
assert silver["camion_id"].str.fullmatch(r"CAM-\d{2}").all()
assert silver["capacidad_kg"].notna().all() and silver["capacidad_kg"].gt(0).all()
assert silver["tipo_camion"].isin(["Refrigerado", "Seco"]).all()
assert set(pendientes["fila_bronze"]) == set(cuarentena_final["fila_bronze"])
assert df_final.loc[df_final["decision_duplicado"].eq("COPIA_EXCLUIDA"), "en_cuarentena_final"].all()
print("Silver:", len(silver), "camiones; cuarentena final:", len(cuarentena_final))
display(acciones.groupby(["codigo_error", "estado_tratamiento"]).size().rename("filas").reset_index())


Silver: 28 camiones; cuarentena final: 4


,codigo_error,estado_tratamiento,filas
0,DUPLICADO_IDENTICO,PENDIENTE,2
1,FORMATO_INVALIDO,PENDIENTE,2


## 5 · Reporte y exportación

El reporte registra el hash del Bronze, las reglas y los resultados reales. La última celda verifica de nuevo el origen y reemplaza las salidas de este notebook.


In [6]:
def crear_reporte():
    datos = [
        ("sha256_bronze", HASH_BRONZE),
        ("version_diagnostico_origen", VERSION_DIAGNOSTICO_REQUERIDA),
        ("version_tratamiento", VERSION_TRATAMIENTO),
        ("filas_bronze", len(bronze)),
        ("filas_cuarentena_inicial", int(df_final["en_cuarentena_inicial"].sum())),
        ("filas_cuarentena_final", len(cuarentena_final)),
        ("filas_liberadas", int((df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).sum())),
        ("filas_silver", len(silver)),
        ("problemas_resueltos", int(acciones["estado_tratamiento"].eq("RESUELTO").sum())),
        ("copias_excluidas", int(acciones["estado_tratamiento"].eq("EXCLUIDO_COMO_COPIA").sum())),
        ("problemas_pendientes", int(acciones["estado_tratamiento"].eq("PENDIENTE").sum())),
    ]
    datos.extend(("regla_" + r["regla_id"], r["estado"]) for _, r in catalogo.iterrows())
    return pd.DataFrame(datos, columns=["metrica", "valor"])

reporte2 = crear_reporte()

def exportar(directorio, tablas):
    if hashlib.sha256(rutas["bronze"].read_bytes()).hexdigest() != HASH_BRONZE:
        raise RuntimeError("El Bronze cambió; se cancela la exportación")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_flota2_",
                                             dir=directorio, encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

tablas = {
    "andinalog_flota_tratado.csv": df_final,
    "andinalog_flota_silver.csv": silver,
    "andinalog_flota_acciones.csv": acciones,
    "andinalog_flota_decisiones_duplicados.csv": decisiones,
    "andinalog_flota_cuarentena_final.csv": cuarentena_final,
    "andinalog_flota_reporte_tratamiento.csv": reporte2,
}
for ruta in exportar(rutas["salidas"], tablas):
    print(ruta)
display(reporte2)


c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_flota\notebook2\salidas\andinalog_flota_tratado.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_flota\notebook2\salidas\andinalog_flota_silver.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_flota\notebook2\salidas\andinalog_flota_acciones.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_flota\notebook2\salidas\andinalog_flota_decisiones_duplicados.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_flota\notebook2\salidas\andinalog_flota_cuarentena_final.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_flota\notebook2\salidas\andinalog_flota_reporte_tratamiento.csv


,metrica,valor
0,sha256_bronze,5da0620ea1883c3bad57ba31f45c24b454fe3dadede3cf...
1,version_diagnostico_origen,GIAD-M3-S4-FLOTA-diagnostico-v2
2,version_tratamiento,GIAD-M3-S4-FLOTA-tratamiento-v1
3,filas_bronze,32
4,filas_cuarentena_inicial,4
5,filas_cuarentena_final,4
6,filas_liberadas,0
7,filas_silver,28
8,problemas_resueltos,0
9,copias_excluidas,0


## Siguiente etapa

Revisa el catálogo y el CSV de acciones antes de afirmar que una fila se trató. Con todas las reglas pendientes, Silver contiene solo los camiones que ya pasaron el diagnóstico sin problemas.
